<a href="https://colab.research.google.com/github/AcSsalazar/the-color-of-emotions/blob/main/Notebooks-EN/3.1-feature-extraction-rav-%26-crema.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Feature Extraction & Data Routing

In this notebook we perform the **feature extraction** process and generate images from two datasets: **RAVDESS** (process detailed in the previous notebook) and **CREMA-D** (*Crowd-sourced Emotional Multimodal Actors Dataset*), by implementing a methodologically rigorous **Data Handling** pipeline for Speech Emotion Recognition (SER).

CREMA-D is a dataset consisting of **7,442 original clips** performed by 91 actors (48 men and 43 women) aged 20 to 74, from diverse ethnicities. The actors recorded a selection of 12 sentences across 6 distinct emotional categories.

You can find the technical details here: [CREMA-D on Kaggle](https://www.kaggle.com/datasets/ejlok1/cremad).

As we will see later, because CREMA-D does not include the *calm* (calma) or *surprised* (sorpresa) emotions, **fundamental decisions** were made for handling these classes during extraction:

1. **Class fusion:** We decided to merge the *calm* class from RAVDESS with the *neutral* class. This is because, in previous classification tests (Machine Learning, Deep Learning, and listening tests with people), the acoustic and perceptual difference between the two is minimal.

2. **Data Augmentation (Surprised):** Since the *surprised* class exists only in RAVDESS, an exclusive **data augmentation** process is applied by adding noise (*AWGN*) and time shifting (*time shift*). The technical details are documented in the corresponding code lines.

### Best practices to avoid overfitting:

1. **Speaker leakage:** Random file splitting (`random_split`) is valid in some contexts, but in our case it carries overfitting risks because if the same actor appears in Train and Test, the Convolutional Neural Network (CNN) will memorize the speaker’s vocal tract instead of the spectral patterns of the emotion. That is why this notebook extracts each actor’s ID and splits at the individual level before processing the audio.

2. **Data leakage via augmentation:** Validation and Test sets must **never** contain synthetic data; evaluating a model on artificially modified audio destroys metric validity. Data augmentation (class `surprised`) is strictly restricted to actors assigned to the training set.

3. **Global Standard Scaling:** Correct use of `StandardScaler` is essential when creating the tabular dataset; if we use global mean and variance we will contaminate the model, because it receives statistical information from the test set during training. The CSV generated here contains pure raw numeric data. Scaling and interpolation (SMOTE) must be applied in the [training notebook](https://github.com/AcSsalazar/the-color-of-emotions/blob/main/Notebooks-EN/5.2-model-training-with-agumentation.ipynb) exclusively on the *Train split*.

In [ ]:
# Imports
#------------------------------------------------------------
import IPython.display as ipd
import librosa
import torch
import shutil
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
import gc, random
import kagglehub
import matplotlib
import os
import logging
import sklearn.preprocessing
#-------------------------------------------------------------
from pandas import DataFrame
from google.colab import drive
from sklearn.preprocessing import StandardScaler
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
#-------------------------------------------------------------
matplotlib.use('Agg')  # No-GUI backend for memory efficiency
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s', stream=sys.stdout, force=True)

In [ ]:
# Download directly from Kaggle
path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")
path_crema = kagglehub.dataset_download("ejlok1/cremad")

Because RAVDESS is recorded at 48 kHz (high fidelity, with energy up to 24 kHz) and CREMA-D at 16 kHz (with a sharp cutoff at 8 kHz), setting a `SAMPLE_RATE` above 16 kHz would mean downsampling RAVDESS and upsampling CREMA-D. That would add synthetic data to our original records. In addition, in this context, a frequency range between 20 Hz and 8000 Hz is more than sufficient, since the human voice concentrates 95% of phonetic and emotional information (fundamental frequency F0 and the first three formants F1, F2, and F3) below 4000 Hz.

In [ ]:
# Configuration variables and paths
#--------------------------------------------------------------------------
FAST_ROOT_DIR = '/content/features_local/ravdess-and-crema-images'
os.makedirs(FAST_ROOT_DIR, exist_ok=True)
# Config
SAMPLE_RATE = 16000 # Limitado por CREMA
MIN_DURATION = 0.5
MAX_DURATION = 3
TARGET_SAMPLES = (SAMPLE_RATE * MAX_DURATION )
HOP_LENGTH = 512
# Mathematical calculation of tensor width (frames)
TARGET_FRAMES = int((SAMPLE_RATE * MAX_DURATION) / HOP_LENGTH) + 1 # ~94 frames
# Features for images
FEATURES_TO_EXTRACT = ['mfcc', 'mel_spec']
IMG_RES = (224, 224)
PAD_MODE = "constant" # Para padding de audios cortos
# Paths Faster
#-------------------------------------------------------------------------
OUT_DIR_CSV = '/content/features_csv_raw.csv'
OUT_DIR_TENSORS = '/content/dataset_split_tensors'
OUT_DIR_IMAGES = '/content/dataset_split_images'

In [ ]:
# Copy to local session for faster processing (SSD)
!cp -r {path}/* {FAST_ROOT_DIR}
!cp -r {path_crema}/* {FAST_ROOT_DIR}

**Attention!** Structurally, RAVDESS comes with a copy of the clips inside the folder: audio_speech_actors_01-24. They must be removed to avoid processing duplicates of these files.

In [ ]:
from genericpath import isdir
target_dir = os.path.join(FAST_ROOT_DIR, "audio_speech_actors_01-24")

if os.path.isdir(target_dir):
  shutil.rmtree(target_dir)
  print("Directory removed")
else:
  ("Directory not found")

In [ ]:
file_list = os.listdir(FAST_ROOT_DIR)
if len(file_list) > 0:
  print("FAST_ROOT_DIR contains files")
  print("-------------------------------")
  print("Puede continuar con el procesamiento")
else:
  print(f"FAST_ROOT_DIR is empty{file_list}")
  print("-------------------------------")
  print("revise path y path_crema")

In [ ]:
# In case of code review and to avoid heavy processing:
# Set False to disable CSV, 3D tensor, and image generation
GENERATE_CSV = False
GENERATE_TENSORS = True
GENERATE_IMAGES = False

### Speaker-Independent Splitting Logic
Before touching a single audio tensor, we analyze the file naming to extract the speaker identity and the emotion. Then, we split the actors mathematically.

In [ ]:
def get_actor_and_emotion(filename):
  """Extracts actor ID and emotion based on strict RAVDESS/CREMA-D naming"""
  if filename.startswith('03'): # Nomenclatura Ravdess (ej. 03-02-XX-01-01-02.wav)
    parts = filename.split('-')
    if len(parts) != 7: return 'unknown'
    actor_id = f"ravdess_{parts[-1].replace('.wav', '')}" # In Python, negative indices count from the end
    rav_map = {1:'neutral', 2:'neutral', 3:'happy', 4:'sad', 5:'angry', 6:'fearful', 7:'disgust', 8:'surprised'}
    emotion = rav_map.get(int(parts[2]), 'unknown')
    return actor_id, emotion

  else: # Nomenclatura CREMA-D (ej. 1001_DFA_ANG_XX.wav)
      parts = filename.split('_')
      if len(parts) < 3: return None, 'unknown'
      actor_id = f"crema_{parts[0]}"
      crema_map = {'NEU':'neutral', 'HAP':'happy', 'SAD':'sad', 'ANG':'angry', 'FEA':'fearful', 'DIS':'disgust'}
      emotion = crema_map.get(parts[2].upper(), 'unknown')
      return actor_id, emotion


# 1. Mapping physical files (copies would be mapped here if we did not remove them)
all_files = [os.path.join(root, f) for root, dirs, files in os.walk(FAST_ROOT_DIR) for f in files if f.endswith('.wav')]
actor_to_files = {}

for f in all_files:
    actor, emo = get_actor_and_emotion(os.path.basename(f))
    if emo == 'unknown': continue
    if actor not in actor_to_files: actor_to_files[actor] = []
    actor_to_files[actor].append(f)

# 2. Stochastic actor split (80% Train, 10% Val, 10% Test)
unique_actors = __builtins__.list(actor_to_files.keys())
random.seed(42) # Immutable seed for consistency between runs
random.shuffle(unique_actors)

train_split = int(0.8 * len(unique_actors))
val_split = int(0.9 * len(unique_actors))

actor_splits = {}
for a in unique_actors[:train_split]: actor_splits[a] = 'train'
for a in unique_actors[train_split:val_split]: actor_splits[a] = 'val'
for a in unique_actors[val_split:]: actor_splits[a] = 'test'

print(f"Total Actores: {len(unique_actors)} | Train: {train_split} | Val: {val_split - train_split} | Test: {len(unique_actors) - val_split}")

## Pure Tabular Extraction (CSV)

**1. The mathematical invariance of time shift (Time Shift)**

In the image pipeline, the spectrogram preserves the temporal dimension on the X axis. A waveform shift moves the pixels to the left or right, forcing the convolutional network (CNN) to be translation-invariant.

* In the CSV pipeline, we collapse the temporal dimension by computing the mean and standard deviation: np.mean(mfccs, axis=1). Mathematically, the mean of a set of values is invariant to the order of those values. If a time shift (np.roll) is applied to the audio signal, the MFCC values change position in time, but their global mean and standard deviation will be virtually identical to those of the original audio (except for tiny edge artifacts).

* If we had included the shift in the CSV, we would have inserted duplicate rows. As discussed before, identical rows in the training set cause overfitting in decision trees because the model memorizes that exact vector.

**2. Feature Space vs. Signal Space (Noise)**

Adding white noise (noise_adder) does change the mean and standard deviation of the MFCCs, generating a different numeric vector. However, in the tabular machine-learning paradigm, altering the raw signal to generate new static rows is an obsolete and inefficient technique for two reasons:

* Distribution control: By injecting acoustic noise, there is no control over how the vector moves in the 125-feature hyperdimensional space. It could generate vectors that overlap with the fearful or angry class, degrading the classifier’s decision boundaries.


* Industry standard (SMOTE): For tabular data, class balancing is done in feature space, not in signal space. Techniques such as SMOTE (Synthetic Minority Over-sampling Technique) take vectors from the minority class (surprised) and mathematically interpolate new points that strictly belong to that class. This yields much cleaner decision boundaries for a Random Forest or an MLP.

**3. Separation of concerns (Data Handling)**

By not including data augmentation in the source CSV, we have created what data engineering calls a Gold Standard Dataset.

* It is an immutable and pure dataset. **It contains exclusively the fundamental truth of the recordings.**

* If we had injected noise during CSV creation for the Train audios, we would have permanently coupled a hyperparameter decision (amount of noise or balancing) to the physical data file.

In [ ]:
# Feature extraction de features dedicada a CSV

def extract_tabular_features(y, sr):
    try:
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=14)[1:,:]
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048)
        mel_spec_db = librosa.power_to_db(mel_spec)
        rmse = librosa.feature.rms(y=y)
        delta = librosa.feature.delta(mfccs)
        delta2 = librosa.feature.delta(mfccs, order=2)

        return np.concatenate([
            np.mean(mfccs, axis=1), np.std(mfccs, axis=1),
            np.mean(delta, axis=1), np.std(delta, axis=1),
            np.mean(delta2, axis=1), np.std(delta2, axis=1),
            [np.mean(rmse), np.std(rmse), np.mean(mel_spec_db), np.std(mel_spec_db)]
        ])
    except Exception:
        return None

if GENERATE_CSV:

  def process_single_file_for_csv(file_path):
      filename = os.path.basename(file_path)
      actor_id, emotion = get_actor_and_emotion(filename)
      split_name = actor_splits.get(actor_id, 'unknown')

      if split_name == 'unknown': return None

      try:
          y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
          y_trimmed, _ = librosa.effects.trim(y, top_db=45) # No se recomienda un parametro menos a 30db
          y_norm = librosa.util.normalize(y_trimmed)
          if (len(y_norm) / sr) < MIN_DURATION: return None

          y_fixed = librosa.util.fix_length(y_norm, size=TARGET_SAMPLES)
          feats = extract_tabular_features(y_fixed, sr)

          if feats is not None:
              return {'features': feats, 'emotion': emotion, 'actor_id': actor_id, 'split': split_name, 'filename': filename}
      except:
          pass
      return None

  print("Starting parallel tabular extraction...")
  all_csv_data = []
  with ProcessPoolExecutor() as executor:
      futures = {executor.submit(process_single_file_for_csv, fp): fp for fp in all_files}
      for future in tqdm(as_completed(futures), total=len(all_files)):
          res = future.result()
          if res: all_csv_data.append(res)

  if all_csv_data:
      columns = ([f'mfcc_mean_{i}' for i in range(13)] + [f'mfcc_std_{i}' for i in range(13)] +
                [f'delta_mean_{i}' for i in range(13)] + [f'delta_std_{i}' for i in range(13)] +
                [f'delta2_mean_{i}' for i in range(13)] + [f'delta2_std_{i}' for i in range(13)] +
                ['rmse_mean', 'rmse_std', 'mel_mean', 'mel_std'])

      df = pd.DataFrame([x['features'] for x in all_csv_data], columns=columns)
      df['emotion'] = [x['emotion'] for x in all_csv_data]
      df['actor_id'] = [x['actor_id'] for x in all_csv_data]
      df['split'] = [x['split'] for x in all_csv_data]
      df['filename'] = [x['filename'] for x in all_csv_data]

      df = df.drop_duplicates(subset=['filename']) # Critical deduplication due to Kaggle's structure
      df.to_csv(OUT_DIR_CSV, index=False)
      print(f"CSV puro guardado en {OUT_DIR_CSV} con {len(df)} filas.")

else:
  print("CSV generation is disabled")

We copy the file to our Drive folder so we do not lose it at the end of the session; it is our backup copy.

In [ ]:
drive.mount('/content/drive')
drive_dir = '/content/drive/MyDrive/features_csv/features_csv_raw.csv'
# Saving CSV a nuestro Drive (opcional)
#!cp  /content/features_csv_raw.csv /content/drive/MyDrive/features_csv

In [ ]:
df =  pd.read_csv(drive_dir)

In [ ]:
print("Contenido del dataframe:")
print("-------------------------")
print(f"{len(df.columns)} Columnas x {len(df)} Filas")

In [ ]:
print("Cantidad de filas por emocion:")
df['emotion'].value_counts()

# 3D Tensor Representation (.PT)

*Processing the acoustic signal purely for a native tensor representation.*

At this stage, we do not address the heuristic visual representation (**PNGs**); instead we process the acoustic signal purely. Feature extraction is performed through digital signal processing (**Librosa/NumPy**) and the data are structured natively as a **rank-3 PyTorch tensor**.

This geometric structure `[Channels, Coefficients, Frames]` mirrors the topology of an RGB image (`[C, H, W]`), allowing us to leverage **Computer Vision** architectures without suffering the lossy compression (*quantization*) of traditional image formats.

## Topological Breakdown of the Tensor

### Dimension 0: Kinematic Channels (Depth = 3)
Unlike a photograph (Red, Green, Blue), our channels encode signal dynamics:

*   **Channel 1 (Static):** The first 13 MFCC coefficients (discarding $C_0$), which model the vocal-tract envelope.
*   **Channel 2 (Velocity):** The Delta coefficient (first-order derivative), which measures the rate of change of MFCCs between contiguous frames.
*   **Channel 3 (Acceleration):** The Delta-Delta coefficient (second-order derivative).

This dimensional superposition guarantees that convolutional filters process the current state of the vocal tract, its velocity, and its mathematical acceleration in a single matrix operation (*dot product*).

### Dimension 1: Cepstral Domain (Height = 13)
Corresponds to the number of retained cepstral coefficients ($C_1$ to $C_{13}$). Unlike a linear spectrogram, this axis does not represent pure frequencies (Hz), but *quefrencies*, decorrelating the signal to isolate emotional timbre beyond the speaker’s base pitch.

### Dimension 2: Temporal Resolution (Width = ~130 Frames)
Represents the temporal evolution of the clip. To guarantee gradient stability in batch training, the raw audio signal is statistically padded/trimmed to a fixed length of 3.0 seconds. With a sampling rate of 22,050 Hz and hop lengths of 512 samples, we obtain a fixed-width matrix, where each column represents a time window of about 23 milliseconds.

![Tensor image](https://kodekloud.com/kk-media/image/upload/v1752883193/notes-assets/images/PyTorch-Introduction-to-PyTorch/pytorch-tensors-vectors-matrices.jpg)

In [ ]:
# Feature extraction dedicated to 3D tensors
# Noise and shift for tensors

def noise_adder(y):
    return y + (0.040 * np.random.uniform() * np.amax(y)) * np.random.normal(size=y.shape[0])

def shift(y):
    return np.roll(y, int(np.random.uniform(low=-5, high=5) * 1000))

def process_single_file_for_tensor(file_path):
    filename = os.path.basename(file_path)
    actor_id, emotion = get_actor_and_emotion(filename)
    split_name = actor_splits.get(actor_id, 'unknown')

    if split_name == 'unknown':
        return None

    try:
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE) # SAMPLE_RATE = 16000
        y_trimmed, _ = librosa.effects.trim(y, top_db=45)

        # Dictionary to store versions of the raw audio
        audio_versions = {'original': y_trimmed}

        # Strict data augmentation injection (same as PNGs)
        if emotion == 'surprised' and split_name == 'train':
            audio_versions['noise'] = noise_adder(y_trimmed)
            audio_versions['shift'] = shift(y_trimmed)

        results = []

        # Process each mathematical version of the audio
        for version_name, y_version in audio_versions.items():
            mfcc = librosa.feature.mfcc(y=y_version, sr=sr, n_mfcc=14, hop_length=HOP_LENGTH)[1:,:]
            delta = librosa.feature.delta(mfcc)
            delta2 = librosa.feature.delta(mfcc, order=2)

            tensor_3d = np.stack([mfcc, delta, delta2], axis=0)

            # Per-channel Z-score normalization
            for i in range(3):
                mean = np.mean(tensor_3d[i])
                std = np.std(tensor_3d[i])
                tensor_3d[i] = (tensor_3d[i] - mean) / (std + 1e-8)

            # Geometric padding/truncating
            current_frames = tensor_3d.shape[2]
            if current_frames < TARGET_FRAMES: # TARGET_FRAMES = ~94
                pad_width = TARGET_FRAMES - current_frames
                tensor_3d = np.pad(tensor_3d, ((0,0), (0,0), (0, pad_width)), mode='constant', constant_values=0)
            elif current_frames > TARGET_FRAMES:
                tensor_3d = tensor_3d[:, :, :TARGET_FRAMES]

            results.append({
                'tensor': tensor_3d.astype(np.float32),
                'emotion': emotion,
                'split': split_name,
                'actor': actor_id
            })

        return results

    except Exception as e:
        print(f"Error processing {filename}: {str(e)}")
        return None

In [ ]:
os.makedirs(OUT_DIR_TENSORS, exist_ok=True)
if GENERATE_TENSORS:
    emotion_to_idx = {
        'angry': 0, 'disgust': 1, 'fearful': 2, 'happy': 3,
        'neutral': 4, 'sad': 5, 'surprised': 6
    }

    print(f"Starting 3D tensor extraction [Canales=3, Frecuencia=13, Tiempo={TARGET_FRAMES}]...")

    data_splits = {'train': [], 'val': [], 'test': []}
    processed_count = 0

    # CRITICAL: We use all_files to avoid data leakage from duplicates
    with ProcessPoolExecutor() as executor:
        futures = {executor.submit(process_single_file_for_tensor, fp): fp for fp in all_files}

        for future in tqdm(as_completed(futures), total=len(all_files), desc="Processing signals"):
            results_list = future.result()
            if results_list:
                # Iterate over the returned list (may have 1 or 3 tensors)
                for res in results_list:
                    tensor_pt = torch.from_numpy(res['tensor'])
                    data_tuple = (tensor_pt, emotion_to_idx[res['emotion']])
                    data_splits[res['split']].append(data_tuple)
                    processed_count += 1

    print(f"\nExtraction finished: {processed_count} tensores generados.")

    print("Serializando datasets en formato binario .pt...")
    for split_name, dataset_list in data_splits.items():
        out_path = os.path.join(OUT_DIR_TENSORS, f"{split_name}_tensors.pt")
        torch.save(dataset_list, out_path)
        print(f"-> Saved {split_name.upper()} ({len(dataset_list)} samples) en {out_path}")

    print("\nDataset tensorial creado exitosamente.")

else:
    print("Tensor generation is disabled")

In [ ]:
generar_copia_tensors = True
if generar_copia_tensors:

  !zip -r -q /content/dataset_split_tensors.zip /content/dataset_split_tensors
  print(f"Archivo comprimido en: content")
else:
   print("Copy disabled")

Tensor preview

In [ ]:
plt.switch_backend("module://matplotlib_inline.backend_inline")
print("Backend activo:", matplotlib.get_backend())

In [ ]:
train_data = torch.load('/content/dataset_split_tensors/train_tensors.pt')
x, y = train_data[0]  # x: [3,13,94]

titles = ["MFCC", "Delta", "Delta-Delta"]
fig, axes = plt.subplots(1, 3, figsize=(15, 3))

for c in range(3):
    im = axes[c].imshow(x[c].numpy(), aspect='auto', origin='lower', cmap='coolwarm')
    axes[c].set_title(f"{titles[c]} | label={y}")
    axes[c].set_xlabel("Time frames")
    axes[c].set_ylabel("Coef")
    plt.colorbar(im, ax=axes[c], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

### 2D Spatial Extraction (Images) with Direct Physical Routing
For 2D-CNN architectures (e.g., EfficientNet), we structure the disk to be consumed natively by PyTorch’s `ImageFolder`.
**Critical rule applied:** Data augmentation (AWGN noise and time shift) is strictly conditioned by `if emotion == 'surprised' and split_name == 'train':`, protecting validation sets against synthetic contamination.

### Cepstral coefficient processing
1. **Omission of the first coefficient:** Coefficient 0 (MFCC-0) represents the continuous energy level (DC offset) or overall volume of the audio frame. Since RAVDESS and CREMA-D audios have different gain levels and microphone distances, MFCC-0 acts as a confounding variable. Dropping it forces the model to focus on spectral shape rather than how loudly the actor shouted relative to the microphone.

2. **Standardization with NumPy:** Instead of normalizing the entire matrix together, we apply standardization (mean 0, variance 1) to each coefficient independently along the time axis (axis=1). This forces subtle variations of MFCC-12 to stand out visually as much as those of MFCC-1. Here, the **epsilon = 1e-8** variable is essential to avoid division-by-zero errors.

In [ ]:
# Signal alteration functions
def noise_adder(y):
    return y + (0.040 * np.random.uniform() * np.amax(y)) * np.random.normal(size=y.shape[0])

def shift(y):
    return np.roll(y, int(np.random.uniform(low=-5, high=5) * 1000))

# Function to save images at 224x224
def save_images(data, sr, out_path, feature_type):
    # Force exact dimensions with no axes or white borders
    fig = plt.figure(figsize=(IMG_RES[0]/100, IMG_RES[1]/100), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis('off')

    if feature_type == 'mel_spec':
        librosa.display.specshow(librosa.power_to_db(data, ref=np.max), sr=sr, cmap='inferno', ax=ax)
    elif feature_type == 'mfcc':
        # TECHNICAL PRECISION: Native standardization with epsilon to avoid division by zero
        # data has shape (coefficients, time frames). We operate along axis 1 (time).
        mean = np.mean(data, axis=1, keepdims=True)
        std = np.std(data, axis=1, keepdims=True)
        epsilon = 1e-8

        data_scaled = (data - mean) / (std + epsilon)
        # Coolwarm offers excellent contrast, mathematically recommended :)
        librosa.display.specshow(data_scaled, sr=sr, cmap='coolwarm', ax=ax)

    plt.savefig(out_path, transparent=True, bbox_inches='tight', pad_inches=0)
    fig.clear()
    plt.close(fig)

def process_single_file_for_images(file_path):
    filename = os.path.basename(file_path)
    actor_id, emotion = get_actor_and_emotion(filename)
    split_name = actor_splits.get(actor_id, 'unknown')

    if split_name == 'unknown': return False

    try:
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
        y_trimmed, _ = librosa.effects.trim(y, top_db=45)
        y_fixed = librosa.util.fix_length(y_trimmed, size=TARGET_SAMPLES)
        audio_versions = {'original': y_fixed}

        # Data leakage prevention: augmentation ONLY in train for surprised
        if emotion == 'surprised' and split_name == 'train':
            audio_versions['noise'] = librosa.util.fix_length(noise_adder(y_trimmed), size=TARGET_SAMPLES)
            audio_versions['shift'] = librosa.util.fix_length(shift(y_trimmed), size=TARGET_SAMPLES)

        prefix = "r_" if filename.startswith('03-') else "c_"

        for feature in FEATURES_TO_EXTRACT:
            # Strict routing for ImageFolder
            emotion_dir = os.path.join(OUT_DIR_IMAGES, feature, split_name, emotion)
            os.makedirs(emotion_dir, exist_ok=True)

            for version_name, y_version in audio_versions.items():
                out_name = f"{prefix}{filename.replace('.wav', '.png')}" if version_name == 'original' else f"{prefix}{version_name}_{filename.replace('.wav', '.png')}"
                out_path = os.path.join(emotion_dir, out_name)

                if os.path.exists(out_path):
                    continue

                if feature == 'mel_spec':
                    data = librosa.feature.melspectrogram(y=y_version, sr=sr, n_mels=128, fmax=8000, fmin=20)
                elif feature == 'mfcc':
                    # We extract 14 coefficients and discard MFCC-0 (index 0)
                    data = librosa.feature.mfcc(y=y_version, sr=sr, n_mfcc=14)[1:, :]

                save_images(data, sr, out_path, feature)

        return True
    except Exception as e:
        print(f"Error processing {filename}: {str(e)}")
        return False

### Preventive Visual Validation

We create a random visualization of four **images** within the dataset to preview the final result before processing the full dataset. In this way, we avoid having to **carry out** the full processing—which takes **around 40 minutes**—only to discover possible errors in the output format. In the worst case, this validation prevents us from having to regenerate the set of **8,882 PNG files**.

In [ ]:
import matplotlib.image as mpimg

# 1. Configuration of the temporary environment
PREVIEW_DIR = '/content/preview_temp'
os.makedirs(PREVIEW_DIR, exist_ok=True)
ORIGINAL_OUT_DIR = OUT_DIR_IMAGES
OUT_DIR_IMAGES = PREVIEW_DIR

# 2. Strategic selection: 1 RAVDESS (ideally surprised) and 1 CREMA-D
ravdess_preview = next((f for f in all_files if '03-01-08' in f), next((f for f in all_files if '03-' in f), None))
crema_preview = next((f for f in all_files if 'c_' in get_actor_and_emotion(os.path.basename(f))[0]), None)
preview_files = [f for f in [ravdess_preview, crema_preview] if f is not None]

print("Generating test images...")
for f in preview_files:
    process_single_file_for_images(f)

# 3. Visualization in Colab
generated_pngs = [os.path.join(root, file) for root, _, files in os.walk(PREVIEW_DIR) for file in files if file.endswith('.png')]

if not generated_pngs:
    print("CRITICAL: No images were generated. Check the error logs.")
else:
    %matplotlib inline
    cols = 4
    rows = (len(generated_pngs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
    axes = axes.flatten() if rows * cols > 1 else [axes]

    for idx, png_path in enumerate(generated_pngs):
        img = mpimg.imread(png_path)
        axes[idx].imshow(img)
        axes[idx].axis('off')

        parts = png_path.replace(PREVIEW_DIR + '/', '').split('/')
        if len(parts) >= 3:
            title = f"{parts[0].upper()} | {parts[1]}\n{parts[2]} | {parts[-1][:12]}..."
            axes[idx].set_title(title, fontsize=9)

    for i in range(len(generated_pngs), len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()
    matplotlib.use('Agg')

# 4. Restore production configuration
OUT_DIR_IMAGES = ORIGINAL_OUT_DIR
print("Validation finished. Environment restored for production.")

## Image generation

In [ ]:
if GENERATE_IMAGES:
    # Safety deduplication (Kaggle bug with RAVDESS)
    unique_files = []
    seen_bases = set()
    for actor, files in actor_to_files.items():
        for f in files:
            base = os.path.basename(f)
            if base not in seen_bases:
                unique_files.append(f)
                seen_bases.add(base)

    print(f"Starting bulk extraction for {len(unique_files)} unique files...")

    processed_count = 0
    with ProcessPoolExecutor() as executor:
        futures = {executor.submit(process_single_file_for_images, fp): fp for fp in unique_files}

        for future in tqdm(as_completed(futures), total=len(unique_files), desc="Generando PNGs"):
            if future.result():
                processed_count += 1
            if processed_count % 500 == 0:
                gc.collect()

    print(f"Extraction completed!")
    print("-------------------------------------------------")
    print(f"{processed_count} files processed successfully.")
    print(f"Directory ready for PyTorch ImageFolder: {OUT_DIR_IMAGES}")
else:
    print("Image generation is disabled (GENERATE_IMAGES = False).")

In [ ]:
os.listdir(OUT_DIR_IMAGES)

In [ ]:
# Saving the result to Drive
generar_copia= False

if generar_copia:

  !zip -r -q /content/dataset_split_img.zip /content/dataset_split_images
  print("compressed file")
  # Move the zip file to Google Drive
  !cp /content/dataset_split_img.zip /content/drive/MyDrive/ravdess_and_crema_images_224_224.zip
  print("Copia de seguridad en Drive completada.")

else:
  print(f"generar_copia: {generar_copia}")
